In [1]:
%load_ext autoreload
%autoreload 2

In [57]:
import numpy as np
import lancedb
import zarr
import numpy as np
import pandas as pd
import uuid
from zarr.storage import ObjectStore
from obstore.store import S3Store
from scipy import sparse

## Data setup

In [62]:
db = lancedb.connect("s3://epiblast/lancezarr_testing/v0_lance/")

In [37]:
num_cells = 1000
median_counts_per_cell = 2_000
num_genes_per_cell = np.random.normal(median_counts_per_cell, scale=(median_counts_per_cell * 0.1), size=num_cells).astype(int)

cell_ranges = np.cumsum(num_genes_per_cell)
cell_ranges = np.insert(cell_ranges, 0, 0)
cell_ranges = np.stack([cell_ranges[:-1], cell_ranges[1:]], axis=1)

dummy_coo_counts = np.random.randint(0, 1000, size=np.sum(num_genes_per_cell).item()).astype(np.int32)
dummy_coo_indices = np.concatenate([
    np.arange(0, n, dtype=np.int32) for n in num_genes_per_cell
], axis=0)

In [45]:
# !aws s3 rm --recursive s3://epiblast/lancezarr_testing/

In [47]:
group_store = ObjectStore(
    S3Store("epiblast", prefix="lancezarr_testing/v0/", region="us-east-2")
)
zarr_group = zarr.create_group(group_store, overwrite=True)

In [51]:
counts_zarray = zarr_group.create_array(
    "counts",
    data=dummy_coo_counts,
    chunks=(5_000,),
    shards=(100_000,),
)
indices_zarray = zarr_group.create_array(
    "indices",
    data=dummy_coo_indices,
    chunks=(5_000,),
    shards=(100_000,),
)

In [61]:
cell_type_choices = ["A-549", "K-562", "HeLa", "Hep-G2", "MCF7", "HNIL", "NGN2"]
metadata_df = pd.DataFrame(
    dict(
        cell_uid=[str(uuid.uuid4()) for _ in range(num_cells)],
        cell_type=np.random.choice(cell_type_choices, size=num_cells),
        start_pos=cell_ranges[:, 0],
        end_pos=cell_ranges[:, 1],
    )
)
cell_table = db.create_table("cells", data=metadata_df)

In [69]:
cell_table.optimize()
cell_table.create_scalar_index("cell_type", replace=True)

## Data loading

In [127]:
from zarrs.utils import ChunkItem, make_chunk_info_for_rust_with_indices
from zarrs.pipeline import get_codec_pipeline_impl

In [132]:
def batch_read_zarr(impl, starts, ends, shard_shape):
    starts = np.asarray(starts, dtype=np.int64)
    ends = np.asarray(ends, dtype=np.int64)
    lengths = ends - starts
    total_rows = int(lengths.sum())
    # shard_rows = shard_shape[0]

    def slice_generator():
        for s, e in zip(starts, ends):
            yield (slice(s, e),)

    # items = []
    # out_offset = 0
    # for i in range(len(starts)):
    #     s = int(starts[i])
    #     e = int(ends[i])
    #     while s < e:
    #         shard_idx = s // shard_rows
    #         local_start = s % shard_rows
    #         chunk_len = min(e, (shard_idx + 1) * shard_rows) - s
    #         items.append(ChunkItem(
    #             key=f"c/{shard_idx}/0",
    #             chunk_subset=[slice(local_start, local_start + chunk_len)],
    #             chunk_shape=shard_shape,
    #             subset=[slice(out_offset, out_offset + chunk_len)],
    #             shape=(total_rows,),
    #         ))
    #         out_offset += chunk_len
    #         s += chunk_len
    items = make_chunk_info_for_rust_with_indices(
        batch_info=slice_generator(),
        drop_axes=(),
        shape=shard_shape,
    )

    out = np.empty((total_rows,), dtype=np.int32)
    impl.retrieve_chunks_and_apply_index(items, out)
    return out, lengths

In [76]:
db = lancedb.connect("s3://epiblast/lancezarr_testing/v0_lance/")
cell_table = db.open_table("cells")

group_store = ObjectStore(
    S3Store("epiblast", prefix="lancezarr_testing/v0/", region="us-east-2")
)
zarr_group = zarr.open(group_store, mode="r")
counts_zarray = zarr_group["counts"]
indices_zarray = zarr_group["indices"]

In [71]:
%%timeit
cell_table.search().where("cell_type == 'A-549'").to_pandas()

26 ms ± 1.15 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [121]:
cell_query_metadata = cell_table.search().where("cell_type IN ('A-549', 'Hep-G2', 'HeLa', 'MCF7', 'NGN2')").to_pandas()

In [122]:
cell_query_metadata.shape

(729, 4)

In [123]:
zarr_impl = get_codec_pipeline_impl(counts_zarray.metadata, counts_zarray.store, strict=True)
zarr_shard_shape = tuple(counts_zarray.metadata.chunk_grid.chunk_shape)

/home/ubuntu/repos/lancell/.venv/lib/python3.13/site-packages/zarrs/pipeline.py:53: RuntimeWarning: Successfully reconstructed a store defined in another Python module. Connection pooling will not be shared across store instances.
  return CodecPipelineImpl(


In [143]:
??counts_zarray.get_block_selection

Signature:
counts_zarray.get_block_selection(
    selection: 'BasicSelection',
    *,
    out: 'NDBuffer | None' = None,
    fields: 'Fields | None' = None,
    prototype: 'BufferPrototype | None' = None,
) -> 'NDArrayLikeOrScalar'
Source:   
    def get_block_selection(
        self,
        selection: BasicSelection,
        *,
        out: NDBuffer | None = None,
        fields: Fields | None = None,
        prototype: BufferPrototype | None = None,
    ) -> NDArrayLikeOrScalar:
        """Retrieve a selection of individual items, by providing the indices
        (coordinates) for each selected item.

        Parameters
        ----------
        selection : int or slice or tuple of int or slice
            An integer (coordinate) or slice for each dimension of the array.
        out : NDBuffer, optional
            If given, load the selected data directly into this buffer.
        fields : str or sequence of str, optional
            For arrays with a structured dtype, one or more

In [138]:
help(counts_zarray)

Help on Array in module zarr.core.array object:

class Array(typing.Generic)
 |  Array(_async_array: 'AsyncArray[T_ArrayMetadata]') -> None
 |
 |  A Zarr array.
 |
 |  Method resolution order:
 |      Array
 |      typing.Generic
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __array__(
 |      self,
 |      dtype: 'npt.DTypeLike | None' = None,
 |      copy: 'bool | None' = None
 |  ) -> 'NDArrayLike'
 |      This method is used by numpy when converting zarr.Array into a numpy array.
 |      For more information, see https://numpy.org/devdocs/user/basics.interoperability.html#the-array-method
 |
 |  __eq__(self, other)
 |      Return self==value.
 |
 |  __getitem__(self, selection: 'Selection') -> 'NDArrayLikeOrScalar'
 |      Retrieve data for an item or region of the array.
 |
 |      Parameters
 |      ----------
 |      selection : tuple
 |          An integer index or slice or tuple of int/slice objects specifying the
 |          requested item or region for each dim

In [133]:
# %%time
out, lengths = batch_read_zarr(
    zarr_impl,
    starts=cell_query_metadata.start_pos.values,
    ends=cell_query_metadata.end_pos.values,
    shard_shape=counts_zarray.shards,
)

ValueError: not enough values to unpack (expected 5, got 1)

In [125]:
out.shape

(1460022,)

In [126]:
lengths

array([2120, 1709, 1894, 1689, 2079, 1987, 1815, 2060, 2538, 2000, 1966,
       2093, 1967, 1962, 2233, 1975, 1918, 2306, 1727, 1918, 2164, 2088,
       2300, 1920, 2154, 2057, 2139, 2041, 2612, 2015, 2115, 1846, 2044,
       2229, 1967, 2226, 2076, 2104, 1800, 1904, 2234, 1788, 1601, 2100,
       1951, 1599, 2071, 2312, 2217, 1618, 2106, 2053, 1938, 1821, 1710,
       1917, 1778, 1958, 2229, 1938, 2125, 1965, 1602, 1988, 2006, 2143,
       2049, 1947, 1795, 2214, 1797, 1873, 2157, 2190, 1917, 1983, 1942,
       1914, 1797, 2290, 2187, 2057, 1959, 1986, 2165, 1782, 1873, 1834,
       2224, 2232, 2232, 2011, 2106, 1747, 2140, 2153, 1901, 1807, 1923,
       1988, 2164, 1980, 2036, 2039, 1887, 1768, 1758, 1931, 1828, 2068,
       1933, 2031, 2240, 1936, 2091, 2036, 1943, 2094, 1754, 2375, 2078,
       2177, 2075, 2203, 2040, 2645, 1981, 2211, 1872, 1990, 2362, 1854,
       1949, 2086, 2539, 1923, 1702, 2107, 2303, 2140, 2277, 1757, 2071,
       1858, 2063, 1807, 1805, 2022, 1697, 1851, 15

In [88]:
return

(array([0, 0, 0, ..., 0, 0, 0], shape=(323549,), dtype=int32),
 array([1894, 1966, 2041, 2015, 2226, 1800, 1904, 1951, 1618, 1710, 1958,
        2229, 2143, 1873, 2157, 2190, 1797, 2187, 2057, 1986, 1834, 2224,
        1747, 1980, 2068, 1936, 1981, 2211, 2086, 2303, 1757, 2063, 1864,
        1943, 1904, 2028, 2438, 2019, 2020, 1960, 1909, 1752, 2268, 2162,
        1769, 2134, 2023, 2012, 2046, 2172, 1600, 1737, 2370, 1819, 2126,
        2129, 1931, 1751, 1944, 2405, 1732, 1864, 1496, 1874, 1712, 1812,
        2067, 1771, 1981, 2028, 1995, 1732, 2122, 1918, 2072, 1983, 2070,
        2074, 1791, 1751, 2303, 2035, 2049, 1934, 1704, 1686, 2023, 1942,
        1925, 2033, 1965, 1951, 1868, 2050, 1971, 1843, 1752, 2139, 2236,
        2070, 2144, 2115, 1941, 2049, 2026, 2112, 1857, 1926, 2056, 2055,
        2002, 1960, 1894, 2205, 2187, 1586, 1941, 1768, 1567, 1947, 1931,
        1940, 2155, 2175, 1922, 2188, 2245, 1980, 1919, 1783, 2029, 1691,
        1643, 2120, 1975, 2002, 1700, 1851, 1802,